# BeExpand - Transformacion de Datos
## Cash Flow Forecasting

**Pipeline:** Carga -> Exploracion -> Clasificacion -> Anomalias -> Normalizacion -> Exportacion

Periodo: Enero 2023 - Abril 2026

---

In [ ]:
# ============================================================
# 1. IMPORTACIONES Y CONFIGURACION
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

from scipy import stats

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Importaciones cargadas correctamente.')

---
## 2. Carga de Datos

Cargamos los 4 archivos CSV desde `data/raw/`.

In [ ]:
# ============================================================
# 2. CARGA DE DATOS
# ============================================================
DATA_DIR = os.path.join('..', 'data', 'raw')

df_crm = pd.read_csv(os.path.join(DATA_DIR, 'crm.csv'), encoding='utf-8-sig')
df_erp = pd.read_csv(os.path.join(DATA_DIR, 'erp.csv'), encoding='utf-8-sig')
df_bank = pd.read_csv(os.path.join(DATA_DIR, 'bank_statements.csv'), encoding='utf-8-sig')
df_external = pd.read_csv(os.path.join(DATA_DIR, 'external_data.csv'), encoding='utf-8-sig')

print('Archivos cargados correctamente:')
print(f'  CRM:      {len(df_crm):>6} registros  -  {df_crm.columns.tolist()}')
print(f'  ERP:      {len(df_erp):>6} registros  -  {df_erp.columns.tolist()}')
print(f'  Banco:    {len(df_bank):>6} registros  -  {df_bank.columns.tolist()}')
print(f'  Externos: {len(df_external):>6} registros  -  {df_external.columns.tolist()}')

---
## 3. Exploracion Inicial

Analizamos tipos de datos, valores nulos, estadisticas basicas antes de transformar.

In [ ]:
print('=== CRM - Informacion General ===')
df_crm.info()
print('\n' + '='*50)
print('=== Valores nulos por columna ===')
print(df_crm.isnull().sum())
print('\n=== Distribucion de estados ===')
print(df_crm['estado'].value_counts())
print('\n=== Valor promedio de proyectos ===')
print(f'Media: {df_crm["valor_proyecto_eur"].mean():,.2f} EUR')
print(f'Mediana: {df_crm["valor_proyecto_eur"].median():,.2f} EUR')
print(f'Total: {df_crm["valor_proyecto_eur"].sum():,.2f} EUR')

In [ ]:
print('=== ERP - Informacion General ===')
df_erp.info()
print('\n=== Valores nulos por columna ===')
print(df_erp.isnull().sum())
print('\n=== Categorias de facturas ===')
print(df_erp['categoria'].value_counts(dropna=False))
print('\n=== Resumen financiero ===')
print(f'Ingresos totales:  {df_erp[df_erp["categoria"]=="ingreso"]["total"].sum():,.2f} EUR')
print(f'Gastos totales:    {df_erp[df_erp["categoria"]!="ingreso"]["total"].sum():,.2f} EUR')
df_erp['fecha_factura'] = pd.to_datetime(df_erp['fecha_factura'])
print(f'\nRango de fechas: {df_erp["fecha_factura"].min().date()} a {df_erp["fecha_factura"].max().date()}')

In [ ]:
print('=== BANK STATEMENTS - Informacion General ===')
df_bank.info()
df_bank['fecha'] = pd.to_datetime(df_bank['fecha'])
print('\n=== Resumen de movimientos ===')
print(f'Total ingresos:    {df_bank[df_bank["tipo"]=="ingreso"]["importe"].sum():,.2f} EUR')
print(f'Total gastos:      {abs(df_bank[df_bank["tipo"]=="gasto"]["importe"].sum()):,.2f} EUR')
print(f'Saldo inicial:     500,000.00 EUR')
print(f'Saldo final:       {df_bank["saldo_acumulado"].iloc[-1]:,.2f} EUR')
print(f'Saldo minimo:      {df_bank["saldo_acumulado"].min():,.2f} EUR')
print(f'Saldo maximo:      {df_bank["saldo_acumulado"].max():,.2f} EUR')
print(f'\nRango de fechas: {df_bank["fecha"].min().date()} a {df_bank["fecha"].max().date()}')

In [ ]:
print('=== DATOS EXTERNOS - Informacion General ===')
df_external.info()
df_external['fecha'] = pd.to_datetime(df_external['fecha'])
print('\n=== Rango de valores macro ===')
print(f'IPC:       {df_external["ipc_interanual_pct"].min():.1f}% a {df_external["ipc_interanual_pct"].max():.1f}%')
print(f'Euribor:   {df_external["euribor_12m_pct"].min():.2f}% a {df_external["euribor_12m_pct"].max():.2f}%')
print(f'Estacional: {df_external["factor_estacional"].min():.2f} a {df_external["factor_estacional"].max():.2f}')

---
## 4. Clasificacion de Transacciones

Enriquecemos los datos con etiquetas utiles para el modelo de forecasting.

- Clasificamos ingresos como recurrentes (retainers) o proyectos
- Agrupamos gastos por categoria (Personal, Estructura, Tecnologia, etc.)
- Clasificamos movimientos bancarios por naturaleza

In [ ]:
# ============================================================
# 4a. Clasificar ingresos del ERP
# ============================================================
df_ingresos = df_erp[df_erp['categoria'] == 'ingreso'].copy()
df_gastos = df_erp[df_erp['categoria'] != 'ingreso'].copy()

def clasificar_ingreso(concepto):
    c = concepto.lower()
    if 'retainer' in c:
        return 'recurrente'
    else:
        return 'proyecto'

df_ingresos['tipo_ingreso'] = df_ingresos['concepto'].apply(clasificar_ingreso)

print('=== Ingresos clasificados ===')
print(df_ingresos['tipo_ingreso'].value_counts())
print()
print(f'Recurrentes: {df_ingresos[df_ingresos["tipo_ingreso"]=="recurrente"]["total"].sum():,.2f} EUR')
print(f'Proyectos:   {df_ingresos[df_ingresos["tipo_ingreso"]=="proyecto"]["total"].sum():,.2f} EUR')

In [ ]:
# ============================================================
# 4b. Clasificar gastos del ERP
# ============================================================
mapa_grupo_gasto = {
    'Nominas':              'Personal',
    'Alquiler oficina':     'Estructura',
    'Suministros':          'Estructura',
    'Software y SaaS':      'Tecnologia',
    'Seguros':              'Estructura',
    'Servicios profesionales': 'Estructura',
    'Material de oficina':  'Estructura',
    'Viajes y dietas':      'Viajes y Ferias',
    'Alquiler stands ferias': 'Viajes y Ferias',
    'Marketing y publicidad': 'Marketing',
    'Traduccion y adaptacion': 'Marketing',
    'Logistica y envios':   'Operaciones',
    'Asesoria legal local': 'Estructura',
    'Hosting y dominios':   'Tecnologia',
    'IVA Trimestral':       'Tributario',
    'Pago Fraccionado IS':  'Tributario',
}
df_gastos['grupo_gasto'] = df_gastos['concepto'].map(mapa_grupo_gasto)
print('=== Gastos por grupo ===')
print(df_gastos.groupby('grupo_gasto')['total'].agg(['sum', 'count']).sort_values('sum', ascending=False))
print()
no_clasif = df_gastos[df_gastos['grupo_gasto'].isna()]['concepto'].unique()
if len(no_clasif) > 0:
    print('ATENCION - Conceptos no clasificados:', no_clasif)

In [ ]:
# ============================================================
# 4c. Clasificar movimientos bancarios
# ============================================================
def clasificar_movimiento(row):
    concepto = str(row['concepto']).lower()
    if 'nomina' in concepto:
        return 'Personal'
    elif 'poliza' in concepto:
        return 'Financiero'
    elif 'iva' in concepto or 'fraccionado' in concepto:
        return 'Tributario'
    elif 'alquiler' in concepto:
        return 'Estructura'
    elif 'feria' in concepto or 'stand' in concepto:
        return 'Viajes y Ferias'
    elif 'cobro' in concepto or 'ingreso' in concepto:
        return 'Cobro Clientes'
    elif 'pago' in concepto:
        return 'Pago Proveedores'
    else:
        return 'Otros'

df_bank['clasificacion'] = df_bank.apply(clasificar_movimiento, axis=1)
print('=== Movimientos bancarios por clasificacion ===')
print(df_bank.groupby('clasificacion')['importe'].agg(['sum', 'count']).sort_values('sum', ascending=False))

---
## 5. Deteccion de Anomalias

Usamos dos metodos estadisticos:
- **Z-score**: valores que se desvian mas de 3 desviaciones tipicas de la media
- **IQR**: valores fuera de 1.5xIQR del rango intercuartilico

Esto simula el paso de IA para deteccion de anomalias del pipeline.

In [ ]:
# ============================================================
# 5a. Anomalias en facturas de ingreso (Z-score)
# ============================================================
df_ingresos['z_score'] = np.abs(stats.zscore(df_ingresos['total']))
df_ingresos['es_anomalia_z'] = df_ingresos['z_score'] > 3
anomalias_ingreso = df_ingresos[df_ingresos['es_anomalia_z']]
print('=== Anomalias en ingresos (Z-score > 3) ===')
print(f'Total facturas: {len(df_ingresos)}')
print(f'Anomalias: {len(anomalias_ingreso)}')
if len(anomalias_ingreso) > 0:
    for _, row in anomalias_ingreso.iterrows():
        print(f'  {row["factura_id"]} | {row["fecha_factura"]} | {row["concepto"][:50]} | {row["total"]:,.2f} EUR | z={row["z_score"]:.2f}')

In [ ]:
# ============================================================
# 5b. Anomalias en gastos (Z-score por grupo)
# ============================================================
def detectar_anomalias_grupo(grupo):
    grupo = grupo.copy()
    if len(grupo) > 3:
        grupo['z_score'] = np.abs(stats.zscore(grupo['total']))
        grupo['es_anomalia'] = grupo['z_score'] > 3
    else:
        grupo['z_score'] = 0
        grupo['es_anomalia'] = False
    return grupo

df_gastos = df_gastos.groupby('grupo_gasto', group_keys=False).apply(detectar_anomalias_grupo)
anomalias_gasto = df_gastos[df_gastos['es_anomalia']]
print('=== Anomalias en gastos (Z-score > 3 por grupo) ===')
print(f'Anomalias detectadas: {len(anomalias_gasto)}')
if len(anomalias_gasto) > 0:
    for _, row in anomalias_gasto.iterrows():
        print(f'  {row["factura_id"]} | {row["grupo_gasto"]:20s} | {row["concepto"]:30s} | {row["total"]:>10,.2f} EUR | z={row["z_score"]:.2f}')

In [ ]:
# ============================================================
# 5c. Anomalias en saldos bancarios (IQR)
# ============================================================
df_bank['anio_mes'] = df_bank['fecha'].dt.to_period('M')
saldos_mensuales = df_bank.groupby('anio_mes')['saldo_acumulado'].last().reset_index()
saldos_mensuales['anio_mes_str'] = saldos_mensuales['anio_mes'].astype(str)

Q1 = saldos_mensuales['saldo_acumulado'].quantile(0.25)
Q3 = saldos_mensuales['saldo_acumulado'].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR
saldos_mensuales['es_anomalia_iqr'] = (saldos_mensuales['saldo_acumulado'] < lim_inf) | (saldos_mensuales['saldo_acumulado'] > lim_sup)

print('=== Anomalias en saldos mensuales (IQR) ===')
print(f'Q1: {Q1:,.0f} EUR  Q3: {Q3:,.0f} EUR  IQR: {IQR:,.0f} EUR')
print(f'Limites: [{lim_inf:,.0f} , {lim_sup:,.0f}] EUR')
anom_saldo = saldos_mensuales[saldos_mensuales['es_anomalia_iqr']]
print(f'Meses anomalos: {len(anom_saldo)} de {len(saldos_mensuales)}')
if len(anom_saldo) > 0:
    for _, row in anom_saldo.iterrows():
        print(f'  {row["anio_mes_str"]}: {row["saldo_acumulado"]:,.2f} EUR')

In [ ]:
# ============================================================
# 5d. Visualizar anomalias
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafico 1: Saldos con anomalias
ax = axes[0]
ax.plot(saldos_mensuales['anio_mes_str'], saldos_mensuales['saldo_acumulado']/1000,
        marker='o', linestyle='-', color='#2196F3', linewidth=2, markersize=6)
anom = saldos_mensuales[saldos_mensuales['es_anomalia_iqr']]
ax.scatter(anom['anio_mes_str'], anom['saldo_acumulado']/1000,
           color='red', s=100, zorder=5, label=f'Anomalias IQR ({len(anom)})')
ax.axhline(y=lim_inf/1000, color='orange', linestyle='--', alpha=0.5, label='Limite inf')
ax.axhline(y=lim_sup/1000, color='orange', linestyle='--', alpha=0.5, label='Limite sup')
ax.set_title('Evolucion Saldo Mensual con Anomalias (IQR)')
ax.set_ylabel('Miles EUR')
ax.legend()
ax.tick_params(axis='x', rotation=45)

# Grafico 2: Boxplot ingresos vs gastos
ax = axes[1]
erp_plot = pd.concat([
    df_ingresos.assign(tipo='Ingresos'),
    df_gastos.assign(tipo='Gastos')
])[['tipo', 'total']]
sns.boxplot(data=erp_plot, x='tipo', y='total', ax=ax, palette=['#4CAF50', '#F44336'])
ax.set_title('Distribucion de Importes: Ingresos vs Gastos')
ax.set_ylabel('Importe (EUR)')

plt.tight_layout()
plt.show()
print(f'Ingresos: {len(anomalias_ingreso)} anomalias | Gastos: {len(anomalias_gasto)} anomalias | Saldos: {len(anom_saldo)} anomalias')

---
## 6. Normalizacion y Consolidacion

Creamos un unico dataset mensual que combine ingresos, gastos, saldo bancario y datos externos.

Este sera el dataset de entrada para el modelo de forecasting (Prophet/LSTM).

In [ ]:
# ============================================================
# 6a. Agregar ingresos por mes
# ============================================================
df_ingresos['anio_mes'] = df_ingresos['fecha_factura'].dt.to_period('M')
df_gastos['anio_mes'] = df_gastos['fecha_factura'].dt.to_period('M')

ingresos_mensuales = df_ingresos.pivot_table(
    values='total', index='anio_mes', columns='tipo_ingreso', aggfunc='sum', fill_value=0
).reset_index()
ingresos_mensuales['total_ingresos'] = ingresos_mensuales['recurrente'] + ingresos_mensuales['proyecto']
print('=== Ingresos mensuales (primeros 5) ===')
print(ingresos_mensuales.head().to_string())

In [ ]:
# ============================================================
# 6b. Agregar gastos por mes y grupo
# ============================================================
gastos_mensuales = df_gastos.pivot_table(
    values='total', index='anio_mes', columns='grupo_gasto', aggfunc='sum', fill_value=0
).reset_index()
cols_gasto = [c for c in gastos_mensuales.columns if c != 'anio_mes']
gastos_mensuales['total_gastos'] = gastos_mensuales[cols_gasto].sum(axis=1)
print('=== Gastos mensuales (primeros 5) ===')
print(gastos_mensuales.head().to_string())

In [ ]:
# ============================================================
# 6c. Saldo bancario al cierre de cada mes
# ============================================================
saldos_cierre = df_bank.groupby('anio_mes').agg({'saldo_acumulado': 'last'}).reset_index()
saldos_cierre.columns = ['anio_mes', 'saldo_cierre_mes']
print('=== Saldos mensuales (primeros 5) ===')
print(saldos_cierre.head().to_string())

In [ ]:
# ============================================================
# 6d. Datos externos
# ============================================================
df_external['anio_mes'] = df_external['fecha'].dt.to_period('M')
print('=== Datos externos (primeros 5) ===')
print(df_external[['anio_mes', 'ipc_interanual_pct', 'euribor_12m_pct', 'factor_estacional']].head().to_string())

In [ ]:
# ============================================================
# 6e. Consolidar en un solo dataset
# ============================================================
df_consolidado = ingresos_mensuales.merge(
    gastos_mensuales, on='anio_mes', how='left'
).merge(
    saldos_cierre, on='anio_mes', how='left'
).merge(
    df_external[['anio_mes', 'ipc_interanual_pct', 'euribor_12m_pct', 'factor_estacional']],
    on='anio_mes', how='left'
)

df_consolidado = df_consolidado.sort_values('anio_mes').reset_index(drop=True)
df_consolidado['cash_flow_neto'] = df_consolidado['total_ingresos'] - df_consolidado['total_gastos']
df_consolidado['ratio_cobertura'] = (df_consolidado['total_ingresos'] / df_consolidado['total_gastos'] * 100).round(1)
df_consolidado['variacion_saldo'] = df_consolidado['saldo_cierre_mes'].diff().fillna(0)

print('=== DATASET CONSOLIDADO ===')
print(f'Dimensiones: {df_consolidado.shape[0]} filas x {df_consolidado.shape[1]} columnas')
print()
print(df_consolidado.head(10).to_string())

In [ ]:
# ============================================================
# 6f. Estadisticas del dataset consolidado
# ============================================================
print('=== ESTADISTICAS DEL DATASET CONSOLIDADO ===')
print(f'Ingresos promedio:      {df_consolidado["total_ingresos"].mean():>10,.2f} EUR/mes')
print(f'Gastos promedio:        {df_consolidado["total_gastos"].mean():>10,.2f} EUR/mes')
print(f'Cash flow neto medio:   {df_consolidado["cash_flow_neto"].mean():>10,.2f} EUR/mes')
print(f'Ratio cobertura medio:  {df_consolidado["ratio_cobertura"].mean():>9.1f}%')
print()
print('Distribucion de ingresos:')
print(f'  Recurrente: {df_consolidado["recurrente"].mean():>10,.2f} EUR/mes')
print(f'  Proyectos:  {df_consolidado["proyecto"].mean():>10,.2f} EUR/mes')
print()
print('Distribucion de gastos:')
for col in cols_gasto:
    media = df_consolidado[col].mean()
    pct = media / df_consolidado['total_gastos'].mean() * 100
    print(f'  {col:25s} {media:>10,.2f} EUR/mes  ({pct:5.1f}%)')

In [ ]:
# ============================================================
# 6g. Visualizar la serie temporal
# ============================================================
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
meses_str = df_consolidado['anio_mes'].astype(str)
x = range(len(df_consolidado))

# G1: Ingresos vs Gastos
ax = axes[0]
ax.fill_between(x, 0, df_consolidado['total_ingresos']/1000, alpha=0.3, color='green', label='Ingresos')
ax.fill_between(x, 0, df_consolidado['total_gastos']/1000, alpha=0.3, color='red', label='Gastos')
ax.plot(x, df_consolidado['total_ingresos']/1000, 'g-o', markersize=4)
ax.plot(x, df_consolidado['total_gastos']/1000, 'r-s', markersize=4)
ax.set_title('Ingresos vs Gastos Mensuales')
ax.set_ylabel('Miles EUR')
ax.legend()
ax.set_xticks(range(0, len(df_consolidado), 6))
ax.set_xticklabels(meses_str[::6], rotation=45)

# G2: Cash Flow Neto
ax = axes[1]
colores = ['green' if v >= 0 else 'red' for v in df_consolidado['cash_flow_neto']]
ax.bar(x, df_consolidado['cash_flow_neto']/1000, color=colores, alpha=0.7)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_title('Cash Flow Neto Mensual')
ax.set_ylabel('Miles EUR')
ax.set_xticks(range(0, len(df_consolidado), 6))
ax.set_xticklabels(meses_str[::6], rotation=45)

# G3: Saldo Bancario
ax = axes[2]
ax.fill_between(x, 0, df_consolidado['saldo_cierre_mes']/1000, alpha=0.3, color='blue')
ax.plot(x, df_consolidado['saldo_cierre_mes']/1000, 'b-o', markersize=4)
ax.set_title('Evolucion del Saldo Bancario')
ax.set_ylabel('Miles EUR')
ax.set_xticks(range(0, len(df_consolidado), 6))
ax.set_xticklabels(meses_str[::6], rotation=45)

plt.tight_layout()
plt.show()
print('Dataset consolidado listo para el modelo de forecasting.')

---
## 7. Exportacion a CSV

Guardamos el dataset consolidado en `data/processed/` para el modelo de forecasting.

In [ ]:
# ============================================================
# 7. EXPORTAR DATASET CONSOLIDADO
# ============================================================
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

output_path = os.path.join(PROCESSED_DIR, 'be_expand_consolidado.csv')
df_consolidado.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'Dataset exportado a: {output_path}')
print(f'Filas: {len(df_consolidado)} | Columnas: {len(df_consolidado.columns)}')
print()
print('Columnas disponibles para el modelo:')
for col in df_consolidado.columns:
    print(f'  - {col}')
print()
print('OK - Transformacion completada. Listo para modelado.')